In [ ]:
%pip install -U "transformers>=4.41" "datasets>=2.20" "accelerate>=0.33" "evaluate>=0.4" sentencepiece sacrebleu tqdm
!pip install sacremoses

In [ ]:
%pip install -U datasets ftfy unidecode langid clean-text[gpl]
# Optional (better LID): fasttext
# %pip install fasttext


In [ ]:
import os, random, numpy as np, torch, inspect
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)
import matplotlib.pyplot as plt

# for reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True

# Project & run knobs (kept small/fast)
PROJECT_DIR = "./marian_bilingual_en_es_pt"
os.makedirs(PROJECT_DIR, exist_ok=True)

MAX_SAMPLES = 1200        # per split, per language direction (train subset)
EVAL_SAMPLES = 300        # validation slice for quick BLEU
MAX_SRC_LEN = 128
MAX_TGT_LEN = 128
BATCH = 8                 # lower to 4/2 if OOM
GRAD_ACCUM = 1
MAX_STEPS = 500           # short fine-tune; bump to 1500+ for quality

device = "cuda" if torch.cuda.is_available() else "cpu" 
#16-bit floating-point formats  
use_bf16 = torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)() #fp 16 has more range but less precision than bf16
use_fp16 = torch.cuda.is_available() and not use_bf16 #use fp16 if bf16 not supported
print(f"Device={device} | bf16={use_bf16} | fp16={use_fp16}") #bf16 preferred when available


In [ ]:
# --- EN→ES (Marian) ---
tok_es = AutoTokenizer.from_pretrained( #   EN→ES (Marian)
    "Helsinki-NLP/opus-mt-en-es", #from hugging face library 
    token=None,               # force anonymous (ignore any stale tokens)
    local_files_only=False,   # allow fresh download if not cached
    use_fast=True
)
mod_es = AutoModelForSeq2SeqLM.from_pretrained(
    "Helsinki-NLP/opus-mt-en-es", #  EN→ES (Marian)
    token=None,
    local_files_only=False
).to(device)

# --- EN→PT (Marian ONLY; no fallback) 

# EN→PT (Marian) 

tok_pt = AutoTokenizer.from_pretrained(
    "Helsinki-NLP/opus-mt-tc-big-en-pt",  
    token=None,
    local_files_only=False,
    use_fast=True
)
mod_pt = AutoModelForSeq2SeqLM.from_pretrained(
    "Helsinki-NLP/opus-mt-tc-big-en-pt",
    token=None,
    local_files_only=False
).to(device)

assert getattr(mod_pt.config, "model_type", "") == "marian", \
    f"Unexpected PT model type: {getattr(mod_pt.config, 'model_type', '?')}"
print("✅ Loaded EN→PT (Marian) from mirror")



In [ ]:
GLOSSARY = {
    "polymerase chain reaction": {"es": "reacción en cadena de la polimerasa", "pt": "reação em cadeia da polimerase"},
    "confidence interval": {"es": "intervalo de confianza", "pt": "intervalo de confiança"},
}
def apply_glossary(text: str, lang: str) -> str:
    out = text
    for en_term, mapping in GLOSSARY.items():
        if lang in mapping:
            out = out.replace(en_term, mapping[lang])
    return out


In [ ]:
def load_small(pair: str, split: str, max_samples: int):
    """
    pair: 'en-es' or 'en-pt'
    returns Dataset with columns: src_text, tgt_text
    """
    ds = load_dataset("opus100", pair, split=split)
    ds = ds.select(range(min(max_samples, len(ds))))
    src_lang, tgt_lang = pair.split("-")
    def norm(ex):
        tr = ex.get("translation", {})
        return {"src_text": tr.get(src_lang, ex.get(src_lang)), "tgt_text": tr.get(tgt_lang, ex.get(tgt_lang))}
    return ds.map(norm, remove_columns=ds.column_names)

train_es = load_small("en-es", "train", MAX_SAMPLES)
valid_es = load_small("en-es", "validation", EVAL_SAMPLES)
train_pt = load_small("en-pt", "train", MAX_SAMPLES)
valid_pt = load_small("en-pt", "validation", EVAL_SAMPLES)

print("Train sizes  EN→ES / EN→PT:", len(train_es), len(train_pt))
print("Valid sizes  EN→ES / EN→PT:", len(valid_es), len(valid_pt))


In [ ]:
def preprocess_any(batch, tok, tgt_short: str):
    # Apply your glossary to targets
    targets = [apply_glossary(t, tgt_short) for t in batch["tgt_text"]]

    # Encode source (Marian: no src/tgt lang flags needed)
    model_inputs = tok(
        batch["src_text"],
        truncation=True,
        max_length=MAX_SRC_LEN,
    )

    # Encode labels (HF>=4.27 supports text_target; keep fallback for older versions)
    try:
        labels = tok(
            text_target=targets,
            truncation=True,
            max_length=MAX_TGT_LEN,
        )
    except TypeError:
        from contextlib import contextmanager
        cm = tok.as_target_tokenizer() if hasattr(tok, "as_target_tokenizer") else contextmanager(lambda: (yield))()
        with cm:
            labels = tok(
                targets,
                truncation=True,
                max_length=MAX_TGT_LEN,
            )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Rebuild tokenized datasets (no NLLB language setting needed)
train_es_tok = train_es.map(lambda b: preprocess_any(b, tok_es, "es"),
                            batched=True, remove_columns=train_es.column_names)
valid_es_tok = valid_es.map(lambda b: preprocess_any(b, tok_es, "es"),
                            batched=True, remove_columns=valid_es.column_names)

train_pt_tok = train_pt.map(lambda b: preprocess_any(b, tok_pt, "pt"),
                            batched=True, remove_columns=train_pt.column_names)
valid_pt_tok = valid_pt.map(lambda b: preprocess_any(b, tok_pt, "pt"),
                            batched=True, remove_columns=valid_pt.column_names)

print("✅ Tokenization done (ES & PT).")


In [ ]:
bleu = evaluate.load("sacrebleu")

def compute_metrics_for(tok):
    def _inner(eval_pred):
        preds, labels = eval_pred
        # decode preds
        decoded_preds = tok.batch_decode(preds, skip_special_tokens=True)
        # replace -100 in labels then decode
        labels = np.where(labels != -100, labels, tok.pad_token_id)
        decoded_labels = tok.batch_decode(labels, skip_special_tokens=True)
        return {"bleu": bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"]}
    return _inner

def make_training_args(**wanted):
    # filter kwargs to support older transformers
    accepted = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
    filtered = {k: v for k, v in wanted.items() if k in accepted}
    dropped = sorted(set(wanted) - set(filtered))
    if dropped: print("Dropped unsupported keys:", dropped)
    return TrainingArguments(**filtered)


In [ ]:
# === Self-healing EN→ES trainer ===
import inspect, torch
from transformers import TrainingArguments, DataCollatorForSeq2Seq, Trainer


# ---- Build/rebuild collator ----
collator_es = DataCollatorForSeq2Seq(
    tokenizer=tok_es,
    model=mod_es,
    padding="longest",               # or pad_to_multiple_of=8 for GPU speed
)

# ---- Version-agnostic TrainingArguments (filters unknown kwargs) ----
def make_training_args(**wanted):
    accepted = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
    filtered = {k: v for k, v in wanted.items() if k in accepted}
    return TrainingArguments(**filtered)

args_es = make_training_args(
    output_dir=f"{PROJECT_DIR}/ckpt_es",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    learning_rate=3e-5,
    lr_scheduler_type="linear",    # ignored if unsupported by your version
    warmup_ratio=0.05,             # ignored if unsupported
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    bf16=use_bf16,
    fp16=use_fp16,
    no_cuda=(device != "cuda"),
    dataloader_num_workers=0,      # safe on Windows
    dataloader_pin_memory=(device == "cuda"),
    evaluation_strategy="no",      # ignored if unsupported
    save_strategy="no",            # ignored if unsupported
)

# ---- Build trainer ----
trainer_es = Trainer(
    model=mod_es,
    args=args_es,
    train_dataset=train_es_tok,
    eval_dataset=None,             # eval later in a separate cell
    data_collator=collator_es,
    tokenizer=tok_es,
)

print("✅ EN→ES trainer ready. Starting training…")
train_output = trainer_es.train()
print("✅ EN→ES done.")


In [ ]:
# === Self-healing EN→PT trainer ===
import inspect, torch
from transformers import TrainingArguments, DataCollatorForSeq2Seq, Trainer

# ---- Build/rebuild collator ----
collator_pt = DataCollatorForSeq2Seq(
    tokenizer=tok_pt,
    model=mod_pt,
    padding="longest",               # or pad_to_multiple_of=8 for GPU speed
)



args_pt = make_training_args(
    output_dir=f"{PROJECT_DIR}/ckpt_pt",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    learning_rate=3e-5,
    lr_scheduler_type="linear",      # ignored if unsupported by your version
    warmup_ratio=0.05,               # ignored if unsupported
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    bf16=use_bf16,
    fp16=use_fp16,
    no_cuda=(device != "cuda"),
    dataloader_num_workers=0,        # safe on Windows
    dataloader_pin_memory=(device == "cuda"),
    evaluation_strategy="no",        # ignored if unsupported
    save_strategy="no",              # ignored if unsupported
)

# ---- Build trainer ----
trainer_pt = Trainer(
    model=mod_pt,
    args=args_pt,
    train_dataset=train_pt_tok,
    eval_dataset=None,               # eval later in a separate cell
    data_collator=collator_pt,
    tokenizer=tok_pt,
)

print("✅ EN→PT trainer ready. Starting training…")
train_output_pt = trainer_pt.train()
print("✅ EN→PT done.")


In [ ]:
# === Accuracy/quality evaluation for EN→ES and EN→PT ===
import torch, numpy as np, evaluate

# Config
EVAL_LIMIT   = 500       # evaluate on up to N examples
MAX_NEW_TOK  = 64
MIN_NEW_TOK  = 3
NUM_BEAMS    = 4
BATCH_SIZE   = 32

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

@torch.inference_mode()
def batch_generate(model, tok, texts, max_len_src=128, max_new=MAX_NEW_TOK, beams=NUM_BEAMS, batch_size=BATCH_SIZE):
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_len_src)
    input_ids = enc["input_ids"]
    attn = enc.get("attention_mask")
    preds = []
    for i in range(0, input_ids.size(0), batch_size):
        sl = slice(i, i + batch_size)
        batch = {"input_ids": input_ids[sl].to(model.device)}
        if attn is not None:
            batch["attention_mask"] = attn[sl].to(model.device)
        out = model.generate(
            **batch,
            max_new_tokens=max_new,
            min_new_tokens=MIN_NEW_TOK,
            num_beams=beams,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            do_sample=False,
        )
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return [p.strip() for p in preds]

def evaluate_pair(model, tok, valid_ds, lang_name="ES"):
    # Slice evaluation set
    src = [r["src_text"] for r in valid_ds][:EVAL_LIMIT]
    refs = [r["tgt_text"] for r in valid_ds][:EVAL_LIMIT]

    # Generate predictions
    preds = batch_generate(model, tok, src, max_len_src=MAX_SRC_LEN)

    # Metrics
    refs_bleu = [[r] for r in refs]  # sacrebleu expects list[list[str]]
    bleu = bleu_metric.compute(predictions=preds, references=refs_bleu)["score"]
    chrf = chrf_metric.compute(predictions=preds, references=refs_bleu)["score"]

    # Exact-match accuracy (strict string equality)
    exact = np.mean([int(p == r) for p, r in zip(preds, refs)]) * 100.0

    # Print a few samples
    k = min(3, len(preds))
    print(f"\n--- {lang_name} samples ---")
    for i in range(k):
        print("SRC:", src[i])
        print("PRD:", preds[i])
        print("REF:", refs[i])
        print("---")

    print(f"{lang_name} -> Metrics on {len(preds)} examples:")
    print(f"  BLEU : {bleu:.2f}")
    print(f"  chrF : {chrf:.2f}")
    print(f"  Exact-match accuracy : {exact:.2f}%")
    return {"bleu": bleu, "chrf": chrf, "exact_acc": exact}

# Evaluate both models if available
results = {}
if 'mod_es' in globals() and 'tok_es' in globals() and 'valid_es' in globals():
    results["es"] = evaluate_pair(mod_es, tok_es, valid_es, lang_name="EN→ES")
if 'mod_pt' in globals() and 'tok_pt' in globals() and 'valid_pt' in globals():
    results["pt"] = evaluate_pair(mod_pt, tok_pt, valid_pt, lang_name="EN→PT")

results


In [ ]:
def plot_loss(trainer, title):
    loss_points = [(e["step"], e["loss"]) for e in getattr(trainer.state, "log_history", []) if "loss" in e]
    if not loss_points:
        print(f"{title}: no loss logs found (set logging_steps > 0).")
        return
    steps, losses = zip(*loss_points)
    plt.figure()
    plt.plot(steps, losses)
    plt.xlabel("Step"); plt.ylabel("Training loss"); plt.title(title); plt.show()

plot_loss(trainer_es, "EN→ES: Training loss")
plot_loss(trainer_pt, "EN→PT: Training loss")


In [ ]:
def translate_es(text, max_new_tokens=128, num_beams=5):
    enc = tok_es([text], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = mod_es.generate(**enc, num_beams=num_beams, max_new_tokens=max_new_tokens, no_repeat_ngram_size=3, early_stopping=True)
    return tok_es.batch_decode(out, skip_special_tokens=True)[0]

def translate_pt(text, max_new_tokens=128, num_beams=5):
    enc = tok_pt([text], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = mod_pt.generate(**enc, num_beams=num_beams, max_new_tokens=max_new_tokens, no_repeat_ngram_size=3, early_stopping=True)
    return tok_pt.batch_decode(out, skip_special_tokens=True)[0]

print(translate_es("this is the systamatic review of a machine translation system that can keep certain equations like log(1+2)=sin(theta) ."
"."))
print(translate_pt("how are u."))


In [ ]:
#ES_CKPT_DIR = r"J:\FINAL PROJECT\marian_bilingual_en_es_pt\en-es_best"
#PT_CKPT_DIR = r"J:\FINAL PROJECT\marian_bilingual_en_es_pt\en-pt_best"

#mod_es.save_pretrained(ES_CKPT_DIR)
#tok_es.save_pretrained(ES_CKPT_DIR)
#mod_pt.save_pretrained(PT_CKPT_DIR)
#tok_pt.save_pretrained(PT_CKPT_DIR)
